# Project FORESIGHT — Inventory Recommendations

## Inventory Recommendation Engine

This notebook uses the output generated by
`05_inventory_risk_analysis.ipynb`.

Input:
`data/processed/inventory_risk_analysis.csv`

Objectives:

- Load inventory risk analysis results
- Inspect available inventory and demand fields
- Calculate inventory position where required
- Calculate target inventory where required
- Calculate recommended replenishment quantity
- Generate inventory recommendations
- Prioritize stockout and overstock situations
- Create SKU-level recommendation summaries
- Export final recommendation files

In [33]:
# Import required libraries

import pandas as pd
import numpy as np
import os

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [44]:
# Load the FORESIGHT analysis-ready dataset

file_path = "../data/processed/analysis_ready.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (20906, 26)


In [45]:
# Convert date column to datetime

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Date conversion completed.")

Date conversion completed.


In [46]:
# Check fields required for inventory recommendations

required_columns = [
    "date",
    "sku_id",
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:")
    for column in missing_columns:
        print("-", column)
else:
    print("All required columns are available.")

All required columns are available.


In [47]:
# Calculate inventory position

df["inventory_position"] = (
    df["on_hand_units"].fillna(0)
    + df["on_order_units"].fillna(0)
)

df[
    [
        "date",
        "sku_id",
        "on_hand_units",
        "on_order_units",
        "inventory_position"
    ]
].head(10)

,date,sku_id,on_hand_units,on_order_units,inventory_position
0,2025-01-01,SKU001,284.0,125,409.0
1,2025-01-02,SKU001,215.0,128,343.0
2,2025-01-03,SKU001,166.0,186,352.0
3,2025-01-04,SKU001,92.0,62,154.0
4,2025-01-05,SKU001,49.0,112,161.0
5,2025-01-06,SKU001,50.0,0,50.0
6,2025-01-07,SKU001,4.0,64,68.0
7,2025-01-08,SKU001,0.0,127,127.0
8,2025-01-09,SKU001,0.0,43,43.0
9,2025-01-10,SKU001,0.0,49,49.0


In [48]:
# Use existing reorder point as target inventory

df["target_inventory"] = (
    df["reorder_point"]
    .fillna(0)
)

print("Target inventory created.")

Target inventory created.


In [49]:
# Calculate recommended replenishment quantity

df["recommended_order_qty"] = (
    df["target_inventory"]
    - df["inventory_position"]
).clip(lower=0)

df["recommended_order_qty"] = (
    df["recommended_order_qty"]
    .round()
    .astype(int)
)

print("Recommended order quantity calculated.")

Recommended order quantity calculated.


In [50]:
# Preview replenishment recommendations

recommendation_preview = df[
    [
        "date",
        "sku_id",
        "on_hand_units",
        "on_order_units",
        "inventory_position",
        "reorder_point",
        "target_inventory",
        "recommended_order_qty"
    ]
].copy()

recommendation_preview.head(20)

,date,sku_id,on_hand_units,on_order_units,inventory_position,reorder_point,target_inventory,recommended_order_qty
0,2025-01-01,SKU001,284.0,125,409.0,255,255,0
1,2025-01-02,SKU001,215.0,128,343.0,255,255,0
2,2025-01-03,SKU001,166.0,186,352.0,255,255,0
3,2025-01-04,SKU001,92.0,62,154.0,255,255,101
4,2025-01-05,SKU001,49.0,112,161.0,255,255,94
5,2025-01-06,SKU001,50.0,0,50.0,255,255,205
6,2025-01-07,SKU001,4.0,64,68.0,255,255,187
7,2025-01-08,SKU001,0.0,127,127.0,255,255,128
8,2025-01-09,SKU001,0.0,43,43.0,255,255,212
9,2025-01-10,SKU001,0.0,49,49.0,255,255,206


In [51]:
# Generate inventory recommendation status

df["recommendation"] = np.where(
    df["recommended_order_qty"] > 0,
    "REPLENISH_STOCK",
    "NO_ACTION"
)

df["recommendation"].value_counts()

recommendation
REPLENISH_STOCK    19208
NO_ACTION           1698
Name: count, dtype: int64

In [53]:
# Assign recommendation priority

df["recommendation_priority"] = np.select(
    [
        df["inventory_position"] < df["reorder_point"],
        df["recommended_order_qty"] > 0
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

df["recommendation_priority"].value_counts()

recommendation_priority
HIGH    19208
LOW      1698
Name: count, dtype: int64

In [54]:
# Create final inventory recommendation dataset

inventory_recommendations = df[
    [
        "date",
        "sku_id",
        "category",
        "subcategory",
        "units_sold",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point",
        "inventory_position",
        "target_inventory",
        "recommended_order_qty",
        "inventory_quality_flag",
        "recommendation_priority",
        "recommendation"
    ]
].copy()

inventory_recommendations.head(20)

,date,sku_id,category,subcategory,units_sold,on_hand_units,on_order_units,lead_time_days,reorder_point,inventory_position,target_inventory,recommended_order_qty,inventory_quality_flag,recommendation_priority,recommendation
0,2025-01-01,SKU001,Furniture,Chair,52.0,284.0,125,6.0,255,409.0,255,0,OK,LOW,NO_ACTION
1,2025-01-02,SKU001,Furniture,Chair,69.0,215.0,128,6.0,255,343.0,255,0,OK,LOW,NO_ACTION
2,2025-01-03,SKU001,Furniture,Chair,49.0,166.0,186,6.0,255,352.0,255,0,OK,LOW,NO_ACTION
3,2025-01-04,SKU001,Furniture,Chair,74.0,92.0,62,6.0,255,154.0,255,101,OK,HIGH,REPLENISH_STOCK
4,2025-01-05,SKU001,Furniture,Chair,43.0,49.0,112,6.0,255,161.0,255,94,OK,HIGH,REPLENISH_STOCK
5,2025-01-06,SKU001,Furniture,Chair,56.0,50.0,0,6.0,255,50.0,255,205,OK,HIGH,REPLENISH_STOCK
6,2025-01-07,SKU001,Furniture,Chair,46.0,4.0,64,6.0,255,68.0,255,187,OK,HIGH,REPLENISH_STOCK
7,2025-01-08,SKU001,Furniture,Chair,51.0,0.0,127,6.0,255,127.0,255,128,OK,HIGH,REPLENISH_STOCK
8,2025-01-09,SKU001,Furniture,Chair,65.0,0.0,43,6.0,255,43.0,255,212,OK,HIGH,REPLENISH_STOCK
9,2025-01-10,SKU001,Furniture,Chair,70.0,0.0,49,6.0,255,49.0,255,206,OK,HIGH,REPLENISH_STOCK


In [55]:
# Export inventory recommendations

output_path = (
    "../data/processed/inventory_recommendations.csv"
)

inventory_recommendations.to_csv(
    output_path,
    index=False
)

print("Inventory recommendations exported successfully.")
print("File:", output_path)

Inventory recommendations exported successfully.
File: ../data/processed/inventory_recommendations.csv


In [56]:
# Generate inventory recommendation status

df["recommendation"] = np.where(
    df["recommended_order_qty"] > 0,
    "REPLENISH_STOCK",
    "NO_ACTION"
)

print("Recommendation status created.")

df["recommendation"].value_counts()

Recommendation status created.


recommendation
REPLENISH_STOCK    19208
NO_ACTION           1698
Name: count, dtype: int64

In [57]:
# Assign recommendation priority

df["recommendation_priority"] = np.select(
    [
        df["inventory_position"] < df["reorder_point"],
        df["recommended_order_qty"] > 0
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

print("Recommendation priority created.")

df["recommendation_priority"].value_counts()

Recommendation priority created.


recommendation_priority
HIGH    19208
LOW      1698
Name: count, dtype: int64

In [58]:
# Create a simple inventory risk status

df["inventory_risk"] = np.select(
    [
        df["inventory_position"] < df["reorder_point"],
        df["inventory_position"] > (
            df["reorder_point"] * 1.5
        )
    ],
    [
        "STOCKOUT_RISK",
        "OVERSTOCK_RISK"
    ],
    default="HEALTHY"
)

print("Inventory risk status created.")

df["inventory_risk"].value_counts()

Inventory risk status created.


inventory_risk
STOCKOUT_RISK     19208
HEALTHY            1033
OVERSTOCK_RISK      665
Name: count, dtype: int64

In [59]:
# Create the final inventory recommendation dataset

inventory_recommendations = df[
    [
        "date",
        "sku_id",
        "category",
        "subcategory",
        "units_sold",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point",
        "inventory_position",
        "target_inventory",
        "recommended_order_qty",
        "inventory_risk",
        "recommendation_priority",
        "recommendation"
    ]
].copy()

inventory_recommendations.head(20)

,date,sku_id,category,subcategory,units_sold,on_hand_units,on_order_units,lead_time_days,reorder_point,inventory_position,target_inventory,recommended_order_qty,inventory_risk,recommendation_priority,recommendation
0,2025-01-01,SKU001,Furniture,Chair,52.0,284.0,125,6.0,255,409.0,255,0,OVERSTOCK_RISK,LOW,NO_ACTION
1,2025-01-02,SKU001,Furniture,Chair,69.0,215.0,128,6.0,255,343.0,255,0,HEALTHY,LOW,NO_ACTION
2,2025-01-03,SKU001,Furniture,Chair,49.0,166.0,186,6.0,255,352.0,255,0,HEALTHY,LOW,NO_ACTION
3,2025-01-04,SKU001,Furniture,Chair,74.0,92.0,62,6.0,255,154.0,255,101,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
4,2025-01-05,SKU001,Furniture,Chair,43.0,49.0,112,6.0,255,161.0,255,94,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5,2025-01-06,SKU001,Furniture,Chair,56.0,50.0,0,6.0,255,50.0,255,205,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
6,2025-01-07,SKU001,Furniture,Chair,46.0,4.0,64,6.0,255,68.0,255,187,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
7,2025-01-08,SKU001,Furniture,Chair,51.0,0.0,127,6.0,255,127.0,255,128,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
8,2025-01-09,SKU001,Furniture,Chair,65.0,0.0,43,6.0,255,43.0,255,212,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
9,2025-01-10,SKU001,Furniture,Chair,70.0,0.0,49,6.0,255,49.0,255,206,STOCKOUT_RISK,HIGH,REPLENISH_STOCK


In [60]:
# Summarize recommendations

recommendation_summary = (
    inventory_recommendations[
        "recommendation"
    ]
    .value_counts()
    .reset_index()
)

recommendation_summary.columns = [
    "Recommendation",
    "Record_Count"
]

recommendation_summary["Percentage"] = (
    recommendation_summary["Record_Count"]
    / len(inventory_recommendations)
    * 100
)

recommendation_summary

,Recommendation,Record_Count,Percentage
0,REPLENISH_STOCK,19208,91.87793
1,NO_ACTION,1698,8.12207


In [61]:
# Summarize inventory risk

risk_summary = (
    inventory_recommendations[
        "inventory_risk"
    ]
    .value_counts()
    .reset_index()
)

risk_summary.columns = [
    "Inventory_Risk",
    "Record_Count"
]

risk_summary["Percentage"] = (
    risk_summary["Record_Count"]
    / len(inventory_recommendations)
    * 100
)

risk_summary

,Inventory_Risk,Record_Count,Percentage
0,STOCKOUT_RISK,19208,91.877930
1,HEALTHY,1033,4.941165
2,OVERSTOCK_RISK,665,3.180905


In [62]:
# Calculate total recommended replenishment quantity

total_recommended_units = (
    inventory_recommendations[
        "recommended_order_qty"
    ].sum()
)

print(
    "Total Recommended Replenishment Units:",
    int(total_recommended_units)
)

Total Recommended Replenishment Units: 4944578


In [63]:
# Create SKU-level inventory recommendation summary

sku_inventory_summary = (
    inventory_recommendations
    .groupby("sku_id")
    .agg(
        average_inventory_position=(
            "inventory_position",
            "mean"
        ),
        average_target_inventory=(
            "target_inventory",
            "mean"
        ),
        total_recommended_order_qty=(
            "recommended_order_qty",
            "sum"
        ),
        risk_records=(
            "inventory_risk",
            lambda x: (x != "HEALTHY").sum()
        )
    )
    .reset_index()
)

sku_inventory_summary.head(20)

,sku_id,average_inventory_position,average_target_inventory,total_recommended_order_qty,risk_records
0,SKU001,104.730769,255.0,82386,544
1,SKU002,51.615970,107.0,31006,524
2,SKU003,21.276557,48.0,14626,541
3,SKU004,113.044706,360.0,112977,420
4,SKU005,160.302198,460.0,163666,545
5,SKU006,24.783883,129.0,56902,546
6,SKU007,158.875458,417.0,141605,544
7,SKU008,186.527473,209.0,31686,388
8,SKU009,31.646520,57.0,13980,515
9,SKU010,96.345216,394.0,158650,533


In [64]:
# Identify SKUs with the highest replenishment requirements

top_replenishment_skus = (
    sku_inventory_summary
    .sort_values(
        "total_recommended_order_qty",
        ascending=False
    )
    .head(20)
)

top_replenishment_skus

,sku_id,average_inventory_position,average_target_inventory,total_recommended_order_qty,risk_records
10,SKU011,196.033771,1231.0,551637,533
22,SKU023,181.626374,821.0,349098,546
17,SKU018,147.260073,786.0,348752,546
24,SKU025,119.514652,714.0,324589,546
19,SKU020,193.996078,774.0,297529,504
13,SKU014,156.070370,690.0,288322,540
34,SKU035,126.413919,604.0,260762,546
27,SKU028,122.353480,483.0,196913,546
26,SKU027,67.141026,391.0,176827,546
4,SKU005,160.302198,460.0,163666,545


In [65]:
# Identify SKUs with the highest number of risk records

top_risk_skus = (
    sku_inventory_summary
    .sort_values(
        "risk_records",
        ascending=False
    )
    .head(20)
)

top_risk_skus

,sku_id,average_inventory_position,average_target_inventory,total_recommended_order_qty,risk_records
5,SKU006,24.783883,129.0,56902,546
24,SKU025,119.514652,714.0,324589,546
27,SKU028,122.353480,483.0,196913,546
35,SKU036,65.327839,264.0,108475,546
34,SKU035,126.413919,604.0,260762,546
26,SKU027,67.141026,391.0,176827,546
17,SKU018,147.260073,786.0,348752,546
22,SKU023,181.626374,821.0,349098,546
4,SKU005,160.302198,460.0,163666,545
29,SKU030,88.582418,235.0,80179,545


In [66]:
# Create processed-data directory if needed

os.makedirs(
    "../data/processed",
    exist_ok=True
)

# Export final recommendations

inventory_recommendations.to_csv(
    "../data/processed/inventory_recommendations.csv",
    index=False
)

print(
    "inventory_recommendations.csv "
    "exported successfully."
)

inventory_recommendations.csv exported successfully.


In [67]:
# Export SKU-level summary

sku_inventory_summary.to_csv(
    "../data/processed/sku_inventory_summary.csv",
    index=False
)

print(
    "sku_inventory_summary.csv "
    "exported successfully."
)

sku_inventory_summary.csv exported successfully.


In [68]:
# Export recommendation and risk summaries

recommendation_summary.to_csv(
    "../data/processed/inventory_recommendation_summary.csv",
    index=False
)

risk_summary.to_csv(
    "../data/processed/inventory_risk_summary.csv",
    index=False
)

print("Summary reports exported successfully.")

Summary reports exported successfully.


In [69]:
# Generate final inventory recommendation metrics

final_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Unique SKUs",
        "Total Recommended Order Units",
        "Stockout Risk Records",
        "Overstock Risk Records",
        "Healthy Records",
        "High Priority Records"
    ],
    "Value": [
        len(inventory_recommendations),

        inventory_recommendations[
            "sku_id"
        ].nunique(),

        int(
            inventory_recommendations[
                "recommended_order_qty"
            ].sum()
        ),

        (
            inventory_recommendations[
                "inventory_risk"
            ] == "STOCKOUT_RISK"
        ).sum(),

        (
            inventory_recommendations[
                "inventory_risk"
            ] == "OVERSTOCK_RISK"
        ).sum(),

        (
            inventory_recommendations[
                "inventory_risk"
            ] == "HEALTHY"
        ).sum(),

        (
            inventory_recommendations[
                "recommendation_priority"
            ] == "HIGH"
        ).sum()
    ]
})

final_summary

,Metric,Value
0,Total Records,20906
1,Unique SKUs,40
2,Total Recommended Order Units,4944578
3,Stockout Risk Records,19208
4,Overstock Risk Records,665
5,Healthy Records,1033
6,High Priority Records,19208


In [70]:
# Show the most important recommendations first

final_priority_view = (
    inventory_recommendations
    .sort_values(
        [
            "recommendation_priority",
            "recommended_order_qty"
        ],
        ascending=[True, False]
    )
)

final_priority_view.head(30)

,date,sku_id,category,subcategory,units_sold,on_hand_units,on_order_units,lead_time_days,reorder_point,inventory_position,target_inventory,recommended_order_qty,inventory_risk,recommendation_priority,recommendation
5440,2025-05-28,SKU011,Small Appliances,Toaster,92.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5471,2025-06-28,SKU011,Small Appliances,Toaster,118.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5685,2026-01-28,SKU011,Small Appliances,Toaster,118.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5744,2026-03-28,SKU011,Small Appliances,Toaster,143.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5761,2026-04-14,SKU011,Small Appliances,Toaster,100.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5836,2026-06-28,SKU011,Small Appliances,Toaster,122.0,0.0,0,14.0,1231,0.0,1231,1231,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5549,2025-09-14,SKU011,Small Appliances,Toaster,113.0,1.0,0,14.0,1231,1.0,1231,1230,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5579,2025-10-14,SKU011,Small Appliances,Toaster,107.0,11.0,0,14.0,1231,11.0,1231,1220,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5379,2025-03-28,SKU011,Small Appliances,Toaster,85.0,28.0,0,14.0,1231,28.0,1231,1203,STOCKOUT_RISK,HIGH,REPLENISH_STOCK
5426,2025-05-14,SKU011,Small Appliances,Toaster,108.0,30.0,0,14.0,1231,30.0,1231,1201,STOCKOUT_RISK,HIGH,REPLENISH_STOCK


# Conclusion

The Inventory Recommendation stage converts the FORESIGHT inventory and demand information into actionable replenishment recommendations.

The final outputs include:

- Inventory Position
- Reorder Point
- Target Inventory
- Recommended Order Quantity
- Inventory Risk
- Recommendation Priority
- Actionable Recommendation
- SKU-level Inventory Summary

The generated CSV files can now be used by the FORESIGHT dashboard and final project documentation.

### Output Files

- `inventory_recommendations.csv`
- `sku_inventory_summary.csv`
- `inventory_recommendation_summary.csv`
- `inventory_risk_summary.csv`

In [71]:
# Validate final inventory recommendations

print("Final dataset shape:", inventory_recommendations.shape)

print("\nMissing values in key recommendation fields:")
print(
    inventory_recommendations[
        [
            "inventory_position",
            "target_inventory",
            "recommended_order_qty",
            "inventory_risk",
            "recommendation"
        ]
    ].isnull().sum()
)

Final dataset shape: (20906, 15)

Missing values in key recommendation fields:
inventory_position       0
target_inventory         0
recommended_order_qty    0
inventory_risk           0
recommendation           0
dtype: int64


In [72]:
# Validate recommended order quantities

print("Minimum recommended order quantity:",
      inventory_recommendations["recommended_order_qty"].min())

print("Maximum recommended order quantity:",
      inventory_recommendations["recommended_order_qty"].max())

print("Total recommended order quantity:",
      inventory_recommendations["recommended_order_qty"].sum())

# Recommended quantity should never be negative
negative_qty = (
    inventory_recommendations["recommended_order_qty"] < 0
).sum()

print("Negative recommendation quantities:", negative_qty)

Minimum recommended order quantity: 0
Maximum recommended order quantity: 1231
Total recommended order quantity: 4944578
Negative recommendation quantities: 0


In [73]:
# Verify exported files

output_files = [
    "../data/processed/inventory_recommendations.csv",
    "../data/processed/sku_inventory_summary.csv",
    "../data/processed/inventory_recommendation_summary.csv",
    "../data/processed/inventory_risk_summary.csv"
]

print("Output file verification:\n")

for file in output_files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌ Missing:", file)

Output file verification:

✅ ../data/processed/inventory_recommendations.csv
✅ ../data/processed/sku_inventory_summary.csv
✅ ../data/processed/inventory_recommendation_summary.csv
✅ ../data/processed/inventory_risk_summary.csv


In [74]:
print("=" * 60)
print("06_inventory_recommendations.ipynb COMPLETED")
print("=" * 60)

print("\nMain output:")
print("inventory_recommendations.csv")

print("\nSupporting outputs:")
print("sku_inventory_summary.csv")
print("inventory_recommendation_summary.csv")
print("inventory_risk_summary.csv")

print("\nInventory recommendation pipeline completed successfully.")

06_inventory_recommendations.ipynb COMPLETED

Main output:
inventory_recommendations.csv

Supporting outputs:
sku_inventory_summary.csv
inventory_recommendation_summary.csv
inventory_risk_summary.csv

Inventory recommendation pipeline completed successfully.


notebooks/
│
├── 01_data_profiling.ipynb
├── 02_eda.ipynb
├── 03_feature_engineering.ipynb
├── 04_demand_forecasting.ipynb
├── 05_inventory_risk_analysis.ipynb
├── 06_inventory_recommendations.ipynb
├── 07_model_evaluation.ipynb
└── 08_final_analysis.ipynb